# VTuber Builder


In [ ]:
import os, pathlib, shutil, subprocess, sys, traceback

# Colab에서 ipywidgets가 실제로 렌더되도록 활성화
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
    from google.colab import files as colab_files
except Exception:
    colab_files = None

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

REPO_URL = "https://github.com/jujumelona/Virtual-pipeline.git"
REPO_DIR = pathlib.Path("/content/Virtual-pipeline")
TRIPOSR_DIR = pathlib.Path("/content/third_party/TripoSR")
TRIPOSR_COMMIT = "107cefdc244c39106fa830359024f6a2f1c78871"
WORK_ROOT = pathlib.Path("/content/vtuber_builder")
INPUT_ROOT = WORK_ROOT / "input"
OUTPUT_ROOT = WORK_ROOT / "output"
INPUT_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

state = {
    "setup_done": False,
    "avatar_vrm": None,
    "result_vrm": None,
    "repo_head": None,
}

ANCHORS = [
    "HEAD_TOP", "FACE", "LEFT_EAR", "RIGHT_EAR", "NECK", "CHEST", "BACK",
    "LEFT_SHOULDER", "RIGHT_SHOULDER", "LEFT_HAND", "RIGHT_HAND",
    "LEFT_FOOT", "RIGHT_FOOT", "HIPS",
]

# ---------- UI ----------
display(HTML("""
<style>
.vtb-card {
  width: min(940px, 96vw);
  box-sizing: border-box;
  border: 1px solid #dadce0;
  border-radius: 14px;
  padding: 18px 20px;
  margin: 8px 0 14px 0;
  background: #fff;
}
.vtb-title { font-size: 28px; font-weight: 750; margin-bottom: 4px; }
.vtb-sub { color: #5f6368; margin-bottom: 12px; }
.vtb-section { font-size: 17px; font-weight: 700; margin: 8px 0 6px; }
.widget-label { font-weight: 600 !important; }
.widget-upload, .widget-dropdown, .widget-toggle-buttons { max-width: 100% !important; }
</style>
"""))

title = widgets.HTML(
    '<div class="vtb-card">'
    '<div class="vtb-title">VTuber Builder</div>'
    '<div class="vtb-sub">모드를 선택하고 파일을 올린 뒤 실행하세요.</div>'
    '</div>'
)

full = widgets.Layout(width="100%")
wide = widgets.Layout(width="100%", max_width="900px")
half = widgets.Layout(width="440px", max_width="100%")
button_layout = widgets.Layout(width="210px", height="42px")

mode = widgets.ToggleButtons(
    options=[("Avatar 만들기", "avatar"), ("Accessory 붙이기", "accessory")],
    value="avatar",
    description="",
    layout=widgets.Layout(width="520px", max_width="100%"),
)

commercial_usage = widgets.Dropdown(
    options=[
        ("기업/수익 사용 허용", "corporation"),
        ("개인 수익 사용", "personalProfit"),
        ("개인 비영리", "personalNonProfit"),
    ],
    value="corporation",
    description="이용 범위",
    layout=half,
    style={"description_width": "110px"},
)

avatar_upload = widgets.FileUpload(
    accept="image/png,image/jpeg,image/webp",
    multiple=False,
    description="캐릭터 이미지 선택",
    layout=widgets.Layout(width="100%", height="42px"),
)

use_recent_avatar = widgets.Checkbox(
    value=True,
    description="방금 만든 Avatar VRM 사용",
    indent=False,
    layout=full,
)

base_vrm_upload = widgets.FileUpload(
    accept=".vrm",
    multiple=False,
    description="Base VRM 선택",
    layout=widgets.Layout(width="100%", height="42px"),
)

accessory_upload = widgets.FileUpload(
    accept="image/png,image/jpeg,image/webp",
    multiple=True,
    description="악세사리 이미지 선택",
    layout=widgets.Layout(width="100%", height="42px"),
)

physics_enabled = widgets.Checkbox(
    value=False,
    description="동적 physics 사용",
    indent=False,
    layout=full,
)

anchor_box = widgets.VBox([], layout=wide)
anchor_selectors = {}

run_button = widgets.Button(
    description="Avatar 생성 실행",
    button_style="primary",
    icon="play",
    layout=button_layout,
)
download_button = widgets.Button(
    description="VRM 다운로드",
    button_style="success",
    icon="download",
    disabled=True,
    layout=button_layout,
)
status_html = widgets.HTML('<span style="color:#5f6368">대기 중</span>')
log_output = widgets.Output(
    layout=widgets.Layout(
        width="100%",
        max_width="900px",
        height="300px",
        overflow_y="auto",
        border="1px solid #dadce0",
        padding="10px",
    )
)

avatar_panel = widgets.VBox([
    widgets.HTML('<div class="vtb-section">캐릭터 이미지</div>'),
    avatar_upload,
    widgets.HTML('<div class="vtb-section">출력 사용 범위</div>'),
    commercial_usage,
], layout=wide)

accessory_panel = widgets.VBox([
    widgets.HTML('<div class="vtb-section">Base VRM</div>'),
    use_recent_avatar,
    base_vrm_upload,
    widgets.HTML('<div class="vtb-section">악세사리 이미지</div>'),
    accessory_upload,
    widgets.HTML('<div class="vtb-section">파일별 부착 위치</div>'),
    anchor_box,
    physics_enabled,
], layout=wide)
accessory_panel.layout.display = "none"

main_card = widgets.VBox([
    mode,
    avatar_panel,
    accessory_panel,
    widgets.HTML("<hr>"),
    widgets.HBox([run_button, download_button], layout=widgets.Layout(gap="12px", flex_flow="row wrap")),
    status_html,
    widgets.HTML('<div class="vtb-section">진행 로그</div>'),
    log_output,
], layout=widgets.Layout(
    width="min(940px, 96vw)",
    border="1px solid #dadce0",
    padding="18px 20px",
))

display(title)
display(main_card)

# ---------- helpers ----------
def upload_entries(upload):
    value = upload.value
    if not value:
        return []
    if isinstance(value, dict):
        rows = []
        for name, item in value.items():
            content = item.get("content", b"") if isinstance(item, dict) else item
            rows.append((name, bytes(content)))
        return rows
    rows = []
    for item in value:
        if isinstance(item, dict):
            rows.append((item.get("name", "upload.bin"), bytes(item.get("content", b""))))
        else:
            rows.append((
                getattr(item, "name", "upload.bin"),
                bytes(getattr(item, "content", b"")),
            ))
    return rows

def save_uploads(upload, directory):
    directory = pathlib.Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    paths = []
    for name, data in upload_entries(upload):
        path = directory / pathlib.Path(name).name
        path.write_bytes(data)
        paths.append(path)
    return paths

def refresh_anchors(*_):
    anchor_selectors.clear()
    rows = []
    for name, _ in upload_entries(accessory_upload):
        selector = widgets.Dropdown(
            options=ANCHORS,
            value="HEAD_TOP",
            description="부착 위치",
            layout=widgets.Layout(width="360px", max_width="100%"),
            style={"description_width": "80px"},
        )
        anchor_selectors[name] = selector
        rows.append(widgets.HBox([
            widgets.HTML(
                f'<div style="width:260px;max-width:45vw;white-space:nowrap;overflow:hidden;text-overflow:ellipsis"><b>{name}</b></div>'
            ),
            selector,
        ], layout=widgets.Layout(flex_flow="row wrap", gap="10px")))
    if not rows:
        rows = [widgets.HTML('<span style="color:#777">악세사리 파일을 선택하면 여기서 각각 부착 위치를 정할 수 있습니다.</span>')]
    anchor_box.children = tuple(rows)

def set_mode(*_):
    is_avatar = mode.value == "avatar"
    avatar_panel.layout.display = "" if is_avatar else "none"
    accessory_panel.layout.display = "none" if is_avatar else ""
    run_button.description = "Avatar 생성 실행" if is_avatar else "Accessory 적용 실행"
    download_button.disabled = True
    state["result_vrm"] = None
    status_html.value = '<span style="color:#5f6368">대기 중</span>'

mode.observe(set_mode, names="value")
accessory_upload.observe(refresh_anchors, names="value")
refresh_anchors()
set_mode()

def run_cmd(cmd):
    print("+", " ".join(str(x) for x in cmd))
    subprocess.run(cmd, check=True)

def ensure_setup():
    if state["setup_done"]:
        return

    if sys.version_info < (3, 12):
        raise RuntimeError(f"Python 3.12+ required, current={sys.version}")

    print("최신 main 준비...")
    if (REPO_DIR / ".git").is_dir():
        run_cmd(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL])
        run_cmd(["git", "-C", str(REPO_DIR), "fetch", "--prune", "origin", "main"])
    else:
        if REPO_DIR.exists():
            shutil.rmtree(REPO_DIR)
        try:
            run_cmd(["git", "clone", "--branch", "main", "--single-branch", REPO_URL, str(REPO_DIR)])
        except subprocess.CalledProcessError as exc:
            raise RuntimeError(
                "Virtual-pipeline public repository를 읽을 수 없습니다. "
                "GitHub에서 repository visibility가 Public인지 확인하세요."
            ) from exc

    run_cmd(["git", "-C", str(REPO_DIR), "checkout", "-B", "main", "origin/main"])
    run_cmd(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/main"])

    head = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
    ).strip()
    origin_head = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "rev-parse", "origin/main"], text=True
    ).strip()
    if head != origin_head:
        raise RuntimeError("latest main sync failed")
    state["repo_head"] = head

    print("TripoSR 준비...")
    TRIPOSR_DIR.parent.mkdir(parents=True, exist_ok=True)
    if (TRIPOSR_DIR / ".git").is_dir():
        run_cmd(["git", "-C", str(TRIPOSR_DIR), "fetch", "--prune", "origin"])
    else:
        if TRIPOSR_DIR.exists():
            shutil.rmtree(TRIPOSR_DIR)
        run_cmd(["git", "clone", "https://github.com/VAST-AI-Research/TripoSR.git", str(TRIPOSR_DIR)])
    run_cmd(["git", "-C", str(TRIPOSR_DIR), "checkout", "--detach", TRIPOSR_COMMIT])

    actual = subprocess.check_output(
        ["git", "-C", str(TRIPOSR_DIR), "rev-parse", "HEAD"], text=True
    ).strip()
    if actual != TRIPOSR_COMMIT:
        raise RuntimeError(f"TripoSR pin mismatch: {actual}")

    print("Dependency 설치...")
    run_cmd([sys.executable, "-m", "pip", "install", "-U", "pip", "setuptools", "wheel"])
    run_cmd([sys.executable, "-m", "pip", "install", "-r", str(TRIPOSR_DIR / "requirements.txt")])
    run_cmd([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR)])

    os.environ["TRIPOSR_DIR"] = str(TRIPOSR_DIR)
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))

    for module_name in list(sys.modules):
        if module_name == "vtuber_pipeline" or module_name.startswith("vtuber_pipeline."):
            del sys.modules[module_name]

    state["setup_done"] = True
    print("준비 완료:", head)

def import_pipeline():
    from vtuber_pipeline.avatar import build_avatar
    from vtuber_pipeline.accessory import reconstruct_accessories, AccessoryPipeline
    return build_avatar, reconstruct_accessories, AccessoryPipeline

def stage_line(name, result):
    if not isinstance(result, dict):
        return
    st = result.get("status", "unknown")
    detail = result.get("error") or result.get("warning") or ""
    mark = "✓" if st == "complete" else ("·" if st in {"cached", "skipped"} else "✗")
    print(f"{mark} {name}: {st}" + (f" — {detail}" if detail else ""))

def run_avatar_mode():
    ensure_setup()
    build_avatar, _, _ = import_pipeline()

    paths = save_uploads(avatar_upload, INPUT_ROOT / "avatar")
    if len(paths) != 1:
        raise ValueError("캐릭터 이미지 1장을 선택하세요.")

    out_dir = OUTPUT_ROOT / "avatar"
    result = build_avatar(
        image_path=str(paths[0]),
        output_dir=str(out_dir),
        config={
            "profile": "commercial",
            "commercial_usage": commercial_usage.value,
        },
    )

    for name, stage in result.get("stages", {}).items():
        stage_line(name, stage)

    if result.get("status") != "complete":
        raise RuntimeError(
            result.get("failed_reason")
            or f"Avatar build failed: {result.get('failed_stages') or result.get('status')}"
        )

    vrm = pathlib.Path(result["vrm_path"])
    if not vrm.is_file():
        raise RuntimeError(f"VRM output missing: {vrm}")

    state["avatar_vrm"] = str(vrm)
    state["result_vrm"] = str(vrm)
    return result

def resolve_base_vrm():
    if use_recent_avatar.value and state.get("avatar_vrm"):
        recent = pathlib.Path(state["avatar_vrm"])
        if recent.is_file():
            return recent

    paths = save_uploads(base_vrm_upload, INPUT_ROOT / "base_vrm")
    if len(paths) != 1:
        raise ValueError("Base VRM 1개를 선택하세요.")
    return paths[0]

def run_accessory_mode():
    ensure_setup()
    _, reconstruct_accessories, AccessoryPipeline = import_pipeline()

    base_vrm = resolve_base_vrm()
    source_paths = save_uploads(accessory_upload, INPUT_ROOT / "accessories")
    if not source_paths:
        raise ValueError("악세사리 이미지를 1장 이상 선택하세요.")

    refresh_anchors()
    reconstructed = reconstruct_accessories(
        [str(p) for p in source_paths],
        str(OUTPUT_ROOT / "accessory_reconstruction"),
        profile="commercial",
    )

    current_vrm = str(base_vrm)
    for index, (source_path, item) in enumerate(zip(source_paths, reconstructed)):
        if item.get("status") != "complete" or not item.get("mesh"):
            raise RuntimeError(
                f"{source_path.name} 3D 재구성 실패: {item.get('error') or item}"
            )

        selector = anchor_selectors.get(source_path.name)
        anchor = selector.value if selector else "HEAD_TOP"
        print(f"[{index+1}/{len(source_paths)}] {source_path.name} → {anchor}")

        result = AccessoryPipeline(str(OUTPUT_ROOT / f"accessory_{index:03d}")).build(
            base_vrm=current_vrm,
            accessory_glb=item["mesh"],
            config={
                "anchor_name": anchor,
                "physics": {"enabled": bool(physics_enabled.value)},
            },
        )
        for stage_name, stage in result.get("stages", {}).items():
            stage_line(f"{source_path.name}/{stage_name}", stage)

        if result.get("status") != "complete":
            raise RuntimeError(
                f"{source_path.name} 적용 실패: "
                f"{result.get('failed_stages') or result.get('incomplete_stages') or result.get('status')}"
            )
        current_vrm = result["output_vrm"]

    final_vrm = pathlib.Path(current_vrm)
    if not final_vrm.is_file():
        raise RuntimeError(f"Combined VRM output missing: {final_vrm}")

    state["result_vrm"] = str(final_vrm)
    return {"status": "complete", "vrm_path": str(final_vrm)}

def on_run(_):
    run_button.disabled = True
    download_button.disabled = True
    state["result_vrm"] = None
    status_html.value = '<b style="color:#1a73e8">실행 중...</b>'

    with log_output:
        clear_output(wait=True)
        try:
            result = run_avatar_mode() if mode.value == "avatar" else run_accessory_mode()
            vrm_path = result.get("vrm_path") or state.get("result_vrm")
            print()
            print("✅ 완료:", vrm_path)
            status_html.value = f'<b style="color:#188038">완료</b> — {vrm_path}'
            download_button.disabled = False
        except Exception as exc:
            print()
            print("❌ 실패:", exc)
            traceback.print_exc()
            status_html.value = f'<b style="color:#d93025">실패</b> — {exc}'
        finally:
            run_button.disabled = False

def on_download(_):
    path = state.get("result_vrm")
    if not path or not pathlib.Path(path).is_file():
        status_html.value = '<b style="color:#d93025">다운로드할 VRM이 없습니다.</b>'
        return
    if colab_files is not None:
        colab_files.download(path)
    else:
        print("결과:", path)

run_button.on_click(on_run)
download_button.on_click(on_download)
